# 02 · Clean flight records and prepare the fact

Keep one row per source record. Read the transformations below, then run them in order. Outputs replace generated data only. The supplied raw CSVs and your Excel workbook remain untouched.

In [ ]:
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
assert (ROOT / "sql" / "01_setup.sql").is_file(), "Open Jupyter from the project root."
DATA = ROOT / "notebooks" / "data"
RAW = DATA / "raw"
OUT = DATA / "processed"
EXPORT = DATA / "exports"
QUALITY = DATA / "quality"
for folder in (OUT, EXPORT, QUALITY):
    folder.mkdir(parents=True, exist_ok=True)

import pyarrow as pa
import pyarrow.parquet as pq


## Make the transformation readable

Keys preserve source lineage, including duplicate operational candidates. Flags are checked before conversion. Only missing attributed-cause minutes become zero. Arrival and departure delays remain nullable. Unavailable arrival outcomes have no delay bucket.

In [ ]:
CAUSE_MAP = {
    "AIR_SYSTEM_DELAY": "air_system_delay_minutes",
    "SECURITY_DELAY": "security_delay_minutes",
    "AIRLINE_DELAY": "airline_delay_minutes",
    "LATE_AIRCRAFT_DELAY": "late_aircraft_delay_minutes",
    "WEATHER_DELAY": "weather_delay_minutes",
}

def clean_chunk(raw, first_row):
    assert raw["CANCELLED"].isin([0, 1]).all(), "Invalid cancellation flag"
    assert raw["DIVERTED"].isin([0, 1]).all(), "Invalid diversion flag"
    dates = pd.to_datetime(dict(year=raw["YEAR"], month=raw["MONTH"], day=raw["DAY"]), errors="raise")
    result = pd.DataFrame({
        "source_row_number": range(first_row, first_row + len(raw)),
        "flight_date": dates,
        "airline_code": raw["AIRLINE"].astype("string").str.strip(),
        "flight_number": raw["FLIGHT_NUMBER"].astype("string").str.strip(),
        "tail_number": raw["TAIL_NUMBER"].astype("string").str.strip(),
        "origin_airport_code": raw["ORIGIN_AIRPORT"].astype("string").str.strip(),
        "destination_airport_code": raw["DESTINATION_AIRPORT"].astype("string").str.strip(),
        "scheduled_departure_hhmm": raw["SCHEDULED_DEPARTURE"].astype("string").str.zfill(4),
        "arrival_delay_minutes": pd.to_numeric(raw["ARRIVAL_DELAY"], errors="raise"),
        "departure_delay_minutes": pd.to_numeric(raw["DEPARTURE_DELAY"], errors="raise"),
        "cancelled": raw["CANCELLED"].astype("int8"),
        "diverted": raw["DIVERTED"].astype("int8"),
        "cancellation_reason_code": raw["CANCELLATION_REASON"].astype("string").str.strip(),
    })
    for original, target in CAUSE_MAP.items():
        result[target] = pd.to_numeric(raw[original], errors="raise").fillna(0.0)
    result["route_code"] = result["origin_airport_code"] + "-" + result["destination_airport_code"]
    hhmm = pd.to_numeric(result["scheduled_departure_hhmm"], errors="raise")
    assert (hhmm // 100).between(0, 23).all() and (hhmm % 100).between(0, 59).all()
    result["departure_hour"] = (hhmm // 100).astype("int8")
    result["peak_period"] = (result["departure_hour"].between(6, 9) | result["departure_hour"].between(16, 19)).astype("int8")
    eligible = result["cancelled"].eq(0) & result["diverted"].eq(0) & result["arrival_delay_minutes"].notna()
    result["delay_bucket"] = pd.cut(
        result["arrival_delay_minutes"], [-float("inf"), 15, 60, float("inf")],
        labels=["On time (<=15m)", "Delay 16-60m", "Severe delay (>60m)"],
    ).astype("string").where(eligible)
    key_fields = ["airline_code", "flight_number", "origin_airport_code",
                  "destination_airport_code", "scheduled_departure_hhmm", "tail_number"]
    candidate = result["flight_date"].dt.strftime("%Y-%m-%d")
    for column in key_fields:
        candidate = candidate + "|" + result[column].fillna("UNKNOWN")
    result["candidate_operational_key"] = candidate
    result["flight_key"] = candidate + "|row=" + result["source_row_number"].astype(str)
    assert result["flight_key"].is_unique
    return result


## Transform in chunks

Parquet is compressed and preserves types. The deterministic sample uses every 128th source row, capped at 50,000. It is a validation sample, not a random estimate. A temporary fact is promoted only after the complete source has been written.

In [ ]:
manifest = json.loads((RAW / "manifest.json").read_text(encoding="utf-8"))
for item in manifest["files"]:
    with (RAW / item["name"]).open("rb") as handle:
        assert hashlib.file_digest(handle, "sha256").hexdigest() == item["sha256"]

temporary = OUT / "fact_flight.tmp.parquet"
writer = None
sample_parts = []
next_row = 1
try:
    for raw in pd.read_csv(RAW / "flights.csv", chunksize=250_000, low_memory=False):
        frame = clean_chunk(raw, next_row)
        table = pa.Table.from_pandas(frame, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(temporary, table.schema, compression="zstd")
        writer.write_table(table)
        sample_parts.append(frame.loc[frame["source_row_number"] % 128 == 0])
        next_row += len(raw)
finally:
    if writer is not None:
        writer.close()
assert next_row - 1 == 5_819_079
assert pq.ParquetFile(temporary).metadata.num_rows == next_row - 1
temporary.replace(OUT / "fact_flight.parquet")
sample = pd.concat(sample_parts, ignore_index=True).head(50_000)
sample.to_csv(OUT / "validation_sample.csv", index=False, lineterminator="\n")
sample.to_csv(OUT / "validation_sample.tsv", sep="\t", index=False, na_rep=r"\N", lineterminator="\n")
assert len(sample) == 45_461
run = {"checked_at_utc": datetime.now(timezone.utc).isoformat(),
       "source_rows": next_row - 1, "curated_rows": next_row - 1,
       "sample_rows": len(sample), "columns": frame.columns.tolist(), "status": "PASS"}
(QUALITY / "transform_run.json").write_text(json.dumps(run, indent=2), encoding="utf-8")
print(f"Fact: {next_row-1:,} rows. Sample: {len(sample):,} rows.")
display(sample.head())


## Export the exact SQL input

This 24-column TSV is also used to compare database content on reruns. `\N` is SQL NULL. Backslash and quote handling must match `sql/02_load.sql`. Keep airport codes and scheduled HHMM as text.

In [ ]:
fact = pq.ParquetFile(OUT / "fact_flight.parquet")
tsv_tmp = OUT / "fact_flight.tmp.tsv"
with tsv_tmp.open("w", encoding="utf-8", newline="") as handle:
    for number, batch in enumerate(fact.iter_batches(batch_size=250_000)):
        batch.to_pandas().to_csv(handle, sep="\t", index=False, header=number == 0,
                                na_rep=r"\N", lineterminator="\n")
tsv_tmp.replace(OUT / "fact_flight.tsv")
print("SQL input:", OUT / "fact_flight.tsv")
print("Column order:", fact.schema_arrow.names)


## Checkpoint

Explain the 24 fields using `reference/data_dictionary.csv`. Identify one cancellation, diversion and early arrival in the sample. The source row suffix prevents silent loss of repeated candidates. Numeric source airport IDs remain unresolved and create separate route labels. Continue to Notebook 03.